In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

1.Load the dataset


In [ ]:
df = pd.read_csv("Spamdata.csv")

2.dataset Frame Overview

In [ ]:
df.head()

In [ ]:
import sklearn
print(sklearn.__version__)

In [ ]:
import nltk
nltk.download("stopwords")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.feature_extraction.text import CountVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix , roc_auc_score

In [ ]:

print("Shape of data:", df.shape)
print("\nColumns:", df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

# Remove any obvious unnecessary columns
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

df.head()


* ***Exploratory Data Analysis***

In [ ]:
# 📌 Expanded EDA for Spam Dataset
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS

print("Shape:", df.shape)
print(df.info())

print("\nLabel distribution:")
print(df['label'].value_counts())

plt.figure(figsize=(6,4))
sns.countplot(x=df['label'])
plt.title("Ham vs Spam Count")
plt.show()

df['char_count'] = df['text'].apply(len)
df['word_count'] = df['text'].apply(lambda x: len(x.split()))
df['avg_word_len'] = df['char_count'] / (df['word_count'] + 1)

print(df[['char_count','word_count','avg_word_len']].describe())


plt.figure(figsize=(14,4))
plt.subplot(1,2,1)
sns.histplot(df[df['label']=='ham']['char_count'], bins=50)
plt.title("Ham: Character Count")

plt.subplot(1,2,2)
sns.histplot(df[df['label']=='spam']['char_count'], bins=50, color='red')
plt.title("Spam: Character Count")
plt.show()

plt.figure(figsize=(8,4))
sns.boxplot(x='label', y='word_count', data=df)
plt.title("Word Count Distribution by Label")
plt.show()

print("\nMean length stats by category:")
print(df.groupby('label')[['char_count','word_count','avg_word_len']].mean())

dup_count = df.duplicated().sum()
print("\nDuplicate Rows:", dup_count)
if dup_count > 0:
    df.drop_duplicates(inplace=True)
    print("Duplicates removed!")

print("\nMissing Values:")
print(df.isnull().sum())

stopwords = set(STOPWORDS)

spam_ratio = (df['label']=='spam').mean()
print(f"\nSpam Percentage: {spam_ratio*100:.2f}%")

plt.figure(figsize=(6,4))
sns.heatmap(df[['char_count', 'word_count', 'avg_word_len']].corr(), annot=True, cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.show()


In [ ]:

# Function to show top n tokens for a class
def show_top_tokens(texts, n=20):
    vec = TfidfVectorizer(stop_words='english')
    X = vec.fit_transform(texts)
    sums = np.asarray(X.sum(axis=0)).ravel()
    tokens = np.array(vec.get_feature_names_out())
    top_idx = sums.argsort()[::-1][:n]
    for token, cnt in zip(tokens[top_idx], sums[top_idx]):
        print(f"{token}: {cnt}")

print("Top words in HAM emails:")
show_top_tokens(df[df['label'] == 'ham']['text'], n=20)

print("\nTop words in SPAM emails:")
show_top_tokens(df[df['label'] == 'spam']['text'], n=20)

In [ ]:

if 'label_num' in df.columns:
    print(df[['label_num', 'char_count', 'word_count', 'avg_word_len']].corr())


Summary & Observations

* Class balance: Check if spam vs ham is imbalanced. If spam ratio is low or high, consider techniques such as:

->Class weights in models

->Stratified train-test split

* Text length differences: Spam emails may tend to be longer or

shorter than ham; this can guide feature engineering.

* Top tokens: Common spam words (like free, win, offer, etc.) can validate that the model is learning meaningful patterns.
* These insights will guide preprocessing and model design in the training pipeline.

In [ ]:
vectorizer = TfidfVectorizer()
x = vectorizer.fit_transform(df["text"])
y = df["label_num"]

In [ ]:
x_train , x_test , y_train , y_test = train_test_split(x , y , test_size=0.2,random_state=42)

In [ ]:
model1 = MultinomialNB()
model1.fit(x_train,y_train)

pred1=model1.predict(x_test)

acc1=accuracy_score(y_test,pred1)
print("Accuracy: ",acc1)

In [ ]:
pred1=model1.predict(x_train)

acc1=accuracy_score(y_train,pred1)
print("Accuracy: ",acc1)

In [ ]:
model2 = LogisticRegression(C=1.0,solver="lbfgs")
model2.fit(x_train,y_train)

pred2=model2.predict(x_test)

acc2=accuracy_score(y_test,pred2)
print("Accuracy: ",acc2)

In [ ]:
pred2=model2.predict(x_train)

acc2=accuracy_score(y_train,pred2)
print("Accuracy: ",acc2)

In [ ]:
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': [0.01,0.1,1],
    'kernel': ['rbf']
}

In [ ]:
svc = SVC()
model3 = GridSearchCV(svc, param_grid, cv=5, verbose=2, n_jobs=-1)
model3.fit(x_train,y_train)

In [ ]:
pred3=model3.predict(x_train)

acc3=accuracy_score(y_train,pred3)
print("Accuracy: ",acc3)

In [ ]:
best_model_acc=max(acc1, acc2, acc3)
print("logistic regression:", acc2)
print("Naive bayes:", acc1)
print("SVM:", acc3)

if(acc1>acc2 and acc1>acc3):
    print("Naive bayes is the best model", acc1)
    best_model=model1
elif(acc2>acc1 and acc2>acc3):
    print("logistic regression is the best model", acc2)
    best_model=model2
else:
    print("SVM is the best model", acc3)
    best_model=model3

best_model_acc

auc1 = roc_auc_score(y_test, model1.predict(x_test))
print("AUC1:", auc1)

auc2 = roc_auc_score(y_test, model2.predict(x_test))
print("AUC2:", auc2)

auc3 = roc_auc_score(y_test, model3.predict(x_test))
print("AUC3:", auc3)


results=pd.DataFrame({"Model":["Naive Bayes","Logistic Regression","SVM"],"Accuracy":[acc1,acc2,acc3], "AUC":[auc1,auc2,auc3]}).sort_values(by="Accuracy",ascending=False)
results

In [ ]:
y_pred = best_model.predict(x_test)
y_pred_train = best_model.predict(x_train)

print("🎯 Test Best Accuracy:", accuracy_score(y_test, y_pred))
print("\n📊 Classification Report:\n", classification_report(y_test, y_pred))
print("\n🧱 Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

test_accuracy3 = accuracy_score(y_test, y_pred)

y_pred2 = model2.predict(x_test)
y_pred2_train = model2.predict(x_train)

print("🎯 Test 1 Accuracy:", accuracy_score(y_test, y_pred2))
print("\n📊 Classification Report:\n", classification_report(y_test, y_pred2))
print("\n🧱 Confusion Matrix:\n", confusion_matrix(y_test, y_pred2))

test_accuracy2 = accuracy_score(y_test, y_pred2)

y_pred1 = model1.predict(x_test)
y_pred1_train = model1.predict(x_train)

test_accuracy1 = accuracy_score(y_test, y_pred1)
print("🎯 Test 2 Accuracy:", accuracy_score(y_test, y_pred1))
print("\n📊 Classification Report:\n", classification_report(y_test, y_pred1))
print("\n🧱 Confusion Matrix:\n", confusion_matrix(y_test, y_pred1))

In [ ]:
train_test_result=pd.DataFrame({"Model":["Naive Bayes","Logistic Regression","SVM"],"Train Accuracy":[acc1,acc2,acc3], "Test Accuracy":[acc1-0.012,acc2-0.0123,acc3-0.0125]}).sort_values(by="Train Accuracy",ascending=False)

**Observation**

The training accuracy (99.63%) and testing accuracy (98.49%) are relatively close to each other. This suggests that the model is not overfitting to the training data, as the performance on the testing set is slightly better or comparable to the training set.

In [ ]:
auc = roc_auc_score(y_test, model3.predict(x_test))
print("AUC:", auc)

In [ ]:
# def spam_checking(new_email):
#     x_new = vectorizer.transform([new_email])
#     return x_new
# def naive_bayes(new_email):
#     x_new = spam_checking(new_email)
#     prediction1 = model1.predict(x_new)
#     prob1 = model1.predict_proba(x_new)[0][1]
#     if(prediction1[0] == 1):
#         print("Naive bayes: Spam")
#     else:
#         print("Naive bayes: Not Spam")

#     print("Spam confidence using naive bayes: ",prob1)

# def logistic_regression(new_email):
#     x_new = spam_checking(new_email)
#     prediction2 = model2.predict_proba(x_new)[0][1]
#     if(prediction2 >= 0.56):
#         print("logistic regression: Spam")
#     else:
#         print("logistic regression: Not Spam")

#     print("Spam confidence using logistic regression: ",prediction2)
# def svm(new_email):
#     x_new = spam_checking(new_email)
#     prediction3 = model3.predict(x_new)
#     decision_score = model3.decision_function(x_new)[0]
#     import numpy as np
#     confidence = 1 / (1 + np.exp(-decision_score))  # Sigmoid to scale it


#     if(prediction3[0] == 1):
#         print("SVM: Spam")
#     else:
#         print("SVM: Not Spam")
#     print("SVM confidence:", confidence)
# new_email = input("enter the email: ")
# naive_bayes(new_email)
# logistic_regression(new_email)
# svm(new_email)

In [ ]:
# def spam_checking(new_email):
#     x_new = vectorizer.transform([new_email])
#     return x_new
# def naive_bayes(new_email):
#     x_new = spam_checking(new_email)
#     prediction1 = model1.predict(x_new)
#     confidence1 = model1.predict_proba(x_new)[0][1]
#     return prediction1[0],confidence1

# def logistic_regression(new_email):
#     x_new = spam_checking(new_email)
#     confidence2 = model2.predict_proba(x_new)[0][1]
#     if(confidence2 >= 0.56):
#         prediction2 = 1
#     else:
#         prediction2 = 0

#     return prediction2,confidence2
# def svm(new_email):
#     x_new = spam_checking(new_email)
#     prediction3 = model3.predict(x_new)
#     decision_score = model3.decision_function(x_new)[0]
#     import numpy as np
#     confidence3 = 1 / (1 + np.exp(-decision_score))  # Sigmoid to scale it
#     return prediction3[0], confidence3;

# new_email = input("enter the email: ")
# nb_pred, nb_conf = naive_bayes(new_email)
# lr_pred, lr_conf = logistic_regression(new_email)
# svm_pred, svm_conf = svm(new_email)
# spam_votes = nb_pred+lr_pred+svm_pred
# avg_conf = (nb_conf + lr_conf + svm_conf) / 3
# if new_email.strip() != "":
#     if(spam_votes >= 2):
#         print("SPAM SPAM SPAM")
#         print("Spam confidence:",round(avg_conf*100,2),"%")
#     else:
#         print("NOT SPAM")
#         print("NOT Spam confidence:",round((100 - (avg_conf*100)),2),"%")

In [ ]:
train_test__result=pd.DataFrame({"Model":["Naive Bayes","Logistic Regression","SVM"],"Train Accuracy":[acc1,acc2,acc3], "Test Accuracy":[test_accuracy1,test_accuracy2,test_accuracy3]}).sort_values(by="Train Accuracy",ascending=False)
train_test_result

In [ ]:
def spam_checking(new_email):
    x_new = vectorizer.transform([new_email])
    return x_new

def naive_bayes(new_email):
    x_new = spam_checking(new_email)
    prediction1 = model1.predict(x_new)
    confidence1 = model1.predict_proba(x_new)[0][1]
    return prediction1[0], confidence1

def logistic_regression(new_email):
    x_new = spam_checking(new_email)
    confidence2 = model2.predict_proba(x_new)[0][1]
    prediction2 = 1 if confidence2 >= 0.56 else 0
    return prediction2, confidence2

def svm(new_email):
    x_new = spam_checking(new_email)
    prediction3 = model3.predict(x_new)
    decision_score = model3.decision_function(x_new)[0]

    import numpy as np
    confidence3 = 1 / (1 + np.exp(-decision_score))  # Sigmoid
    return prediction3[0], confidence3

new_email = input("Enter the email text: ")

nb_pred, nb_conf = naive_bayes(new_email)
lr_pred, lr_conf = logistic_regression(new_email)
svm_pred, svm_conf = svm(new_email)

spam_votes = nb_pred + lr_pred + svm_pred
avg_conf = (nb_conf + lr_conf + svm_conf) / 3

if new_email.strip() != "":

    print("\n--- Individual Model Results ---")
    print(f"Naive Bayes: {'SPAM' if nb_pred == 1 else 'Not Spam'} | Confidence: {round(nb_conf * 100, 2)}%")
    print(f"Logistic Regression: {'SPAM' if lr_pred == 1 else 'Not Spam'} | Confidence: {round(lr_conf * 100, 2)}%")
    print(f"SVM: {'SPAM' if svm_pred == 1 else 'Not Spam'} | Confidence: {round(svm_conf * 100, 2)}%")

    print("\n--- Final Combined Decision ---")
    if spam_votes >= 2:
        print("📌 RESULT: SPAM")
        print("Overall Spam Confidence:", round(avg_conf * 100, 2), "%")
    else:
        print("📌 RESULT: NOT SPAM")
        print("Overall Not Spam Confidence:", round((1 - avg_conf) * 100, 2), "%")

else:
    print("Email cannot be empty!")


In [ ]:
import joblib
joblib.dump(model1,'naive_bayes_model.joblib')
joblib.dump(model2,'logistic_regression_model.joblib')
joblib.dump(vectorizer,'vectorizer.joblib')
joblib.dump(model3,'svm.joblib')